# 04 – DBSCAN (M2)

Phụ trách: M2 (Chu Thành Tân). Theo checklist M2, Phần 4:
- **4.1 Đồ thị k-distance có trọng số**, rút ra các eps ứng viên.
- **4.2 So sánh chạy trên từng chuyến (`full`) với chạy có trọng số (`weighted`)**, đo runtime và RAM.
- **4.3 Lưới eps × min_samples** (cách có trọng số), lưu `dbscan_tuning.csv`.
- **4.4 So sánh các cấu hình ứng viên** trên bảng và bản đồ, đề xuất cấu hình; sau khi chốt thì ghi log và vẽ bản đồ Folium.

Cách chạy đã được duyệt, khác spec (`docs/Phan 4/4.0-quyet-dinh-dbscan.md`):
1. Fit trên toạ độ duy nhất với `sample_weight` = số chuyến.
2. `min_samples` theo tỷ lệ số chuyến của lát cắt, kèm 5 và 20 của spec để đối chiếu.
3. Eps 500–3000 m, tìm điểm gãy trên k-distance có trọng số.

Dữ liệu đang dùng: **bản đầy đủ v1 do M1 phát hành** (`master_v1`, 4 tuần 17/04–14/05/2023). Đổi `DATA_KEY = "model_input"` nếu muốn chạy trên mẫu phân tầng 100k (`sample_v1`).

In [ ]:
# ===== O THIET LAP (chep nguyen o nay vao dau moi notebook) =====
# Colab: clone repo, cai thu vien, them goc repo vao sys.path.
# May ca nhan: tu tim goc repo, khong cai gi them.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if IN_COLAB:
    from src.colab import link_drive_data
    link_drive_data("KhaiPhaDuLieu")

print("Colab:", IN_COLAB, "| Goc repo:", ROOT)

## Tham số

In [ ]:
import hashlib, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from src.config import load_config, resolve_path
from src.models.kmeans import aggregate_unique
from src.models.dbscan import weighted_k_distance, k_distance_curve, min_samples_from_frac, knee_eps

cfg = load_config()
DATA_KEY = "master"              # "master" (master_v1), "model_input" (sample_v1), hoac "pilot_m2"
MANIFEST_KEY = {"pilot_m2": "pilot_m2_manifest", "master": "master_manifest", "model_input": "manifest"}[DATA_KEY]
SLICES = list(cfg["time_slices"])
DB = cfg["dbscan"]
data_path = resolve_path(cfg, DATA_KEY)
man = json.loads(resolve_path(cfg, MANIFEST_KEY).read_text(encoding="utf-8"))
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == man["sha256"], "sha256 khong khop manifest!"
df = pd.read_parquet(data_path)
SAMPLE_ID = man["sample_id"]

def k_list(n_trips):
    """(nhan, k): k theo spec va k theo ty le so chuyen cua lat cat."""
    ks = [(f"spec {m}", m) for m in DB["min_samples"]]
    ks += [(f"{f * 100:g}% = {min_samples_from_frac(n_trips, f)}", min_samples_from_frac(n_trips, f))
           for f in DB["min_samples_frac"]]
    return ks

for s in SLICES:
    n = int((df.time_slice == s).sum())
    print(f"{s:11s} {n:>6,} chuyen -> k:", [k for _, k in k_list(n)])

## 4.1 Khoảng cách giữa các toạ độ

Khoảng cách tới toạ độ gần nhất cho biết eps phải lớn khoảng bao nhiêu thì các toạ độ mới bắt đầu nối với nhau.

In [ ]:
spacing = []
for s in SLICES:
    Xu, w, _ = aggregate_unique(df.loc[df.time_slice == s, ["x_m", "y_m"]].to_numpy("float64"))
    nn = NearestNeighbors(n_neighbors=2).fit(Xu).kneighbors(Xu)[0][:, 1]
    spacing.append({"time_slice": s, "n_coords": len(Xu), "nn_p10_m": np.percentile(nn, 10), "nn_median_m": np.median(nn),
                    "nn_p90_m": np.percentile(nn, 90), "nn_max_m": nn.max()})
pd.DataFrame(spacing).round(0)

## 4.1 Đồ thị k-distance có trọng số

Với mỗi toạ độ, k-distance có trọng số là **khoảng cách nhỏ nhất cần đi để gom đủ k chuyến** (tính cả chính nó). Đó chính là ngưỡng eps để toạ độ đó trở thành core. Đồ thị vẽ theo từng chuyến: trục ngang là tỷ lệ chuyến tích luỹ, trục dọc là k-distance.

- `self_core_pct`: % chuyến ở toạ độ có ≥ k chuyến, tức là tự là core ở mọi eps.
- `p50/p75/p90`: eps cần để 50/75/90% chuyến nằm ở toạ độ core.
- `knee_eps`: điểm gãy gợi ý, tính trên phần khoảng cách > 0.

In [ ]:
rows, curves = [], {}
for s in SLICES:
    X = df.loc[df.time_slice == s, ["x_m", "y_m"]].to_numpy("float64")
    Xu, w, _ = aggregate_unique(X)
    for label, k in k_list(len(X)):
        share, d = k_distance_curve(X, k)
        curves[(s, label)] = (share, d)
        kd = weighted_k_distance(Xu, w, k)
        pct = lambda q: float(d[np.searchsorted(share, q)])
        try:
            knee = knee_eps(share, d)
        except ValueError:
            knee = np.nan
        rows.append({"time_slice": s, "k_label": label, "k": k, "n_trips": len(X),
                     "self_core_pct": round(w[kd == 0].sum() / w.sum() * 100, 1),
                     "p50_m": pct(0.5), "p75_m": pct(0.75), "p90_m": pct(0.9), "knee_eps_m": knee})
kdist = pd.DataFrame(rows).assign(sample_id=SAMPLE_ID)
kd_path = resolve_path(cfg, "run_log").parent / f"dbscan_kdist_{SAMPLE_ID}.csv"
kdist.to_csv(kd_path, index=False)
print("Da luu:", kd_path.relative_to(cfg["_root"]))
kdist.drop(columns=["sample_id", "n_trips"]).round(0)

In [ ]:
fig, axes = plt.subplots(1, len(SLICES), figsize=(5.2 * len(SLICES), 4.2), sharey=True)
for ax, s in zip(axes, SLICES):
    for (ss, label), (share, d) in curves.items():
        if ss != s: continue
        ls = "--" if label.startswith("spec") else "-"
        ax.step(share * 100, d, where="post", ls=ls, lw=1.4, label=f"k {label}")
        knee = kdist.loc[(kdist.time_slice == s) & (kdist.k_label == label), "knee_eps_m"].iloc[0]
        if np.isfinite(knee):
            ax.scatter([share[np.searchsorted(d, knee)] * 100], [knee], s=18, zorder=3)
    for e in DB["eps_m"]:
        ax.axhline(e, color="grey", lw=.5, alpha=.4)
    ax.set_title(f"{s} (n={int((df.time_slice == s).sum()):,})")
    # 79-100% chuyen o toa do tu la core (k-distance = 0): phong to phan con lai
    x0 = max(0, kdist.loc[kdist.time_slice == s, "self_core_pct"].min() - 3)
    ax.set_xlim(x0, 100.3)
    ax.set_xlabel(f"Ty le chuyen tich luy (%) - phong to tu {x0:.0f}%"); ax.grid(alpha=.3); ax.set_ylim(0, 4000)
axes[0].set_ylabel("k-distance co trong so (m)")
axes[-1].legend(fontsize=7, loc="upper left")
fig.suptitle(f"DBSCAN: k-distance co trong so - {SAMPLE_ID} (net dut = min_samples theo spec; vach xam = eps trong config)")
fig.tight_layout()
fig_path = resolve_path(cfg, "figures_dir") / f"dbscan_kdist_{SAMPLE_ID}.png"
fig.savefig(fig_path, dpi=150)
print("Da luu:", fig_path.relative_to(cfg["_root"]))

### Eps ứng viên

Gom các điểm gãy (làm tròn tới 50 m) theo từng k; dùng làm lưới eps mịn cho mục 4.3, cộng thêm lưới thô trong config.

In [ ]:
knees = kdist.dropna(subset=["knee_eps_m"]).assign(knee_r=lambda d: (d.knee_eps_m / 50).round() * 50)
print("Diem gay (m) theo k va lat cat:")
print(knees.pivot_table(index="k_label", columns="time_slice", values="knee_r", aggfunc="first").to_string())
cand = sorted(set(knees.knee_r.astype(int)) | set(DB["eps_m"]))
print("\nEps ung vien cho 4.3:", cand)

## 4.2 So sánh chạy trên từng chuyến (`full`) với chạy có trọng số (`weighted`)

- Lát cắt WEEKEND (lớn nhất), mẫu con tăng dần theo `config/experiment.yaml` (`pilot_sizes`: 10k → 30k → 50k) → toàn bộ, lấy ngẫu nhiên có seed.
- 3 cấu hình đại diện: (eps 500 m, 0,5%), (eps 1000 m, 0,5%), (eps 1500 m, spec 20). `min_samples` theo tỷ lệ tính trên cỡ mẫu con.
- **Bảo vệ RAM:** trước mỗi lần chạy `full`, ước tính số cặp láng giềng bằng `estimate_full_neighbors` (đã kiểm tra khớp chính xác với số đếm thật). Bỏ qua lần chạy nếu ước tính vượt `MAX_FULL_GB`, hoặc nếu RAM hệ thống đã vượt `ram_stop_ratio` (85%) trong config.
- So sánh: ARI giữa hai cách, số cụm, Noise Ratio, tập điểm nhiễu có trùng khớp không; runtime và RAM thêm (`mem_delta_mb`) đo bằng `measure`.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score
from src.evaluation import measure, system_ram_ratio
from src.models.dbscan import estimate_full_neighbors, fit_weighted
from src.m2_pilot import subsample

MAX_FULL_GB = 4.5
SIZES = list(cfg["sampling"]["pilot_sizes"]) + [None]   # 10k, 30k, 50k va toan bo lat cat
CONFIGS = [(500, ("frac", 0.005)), (1000, ("frac", 0.005)), (1500, ("spec", 20))]
part_all = df[df.time_slice == "WEEKEND"].reset_index(drop=True)

rows = []
for n in SIZES:
    part = subsample(part_all, n if n else len(part_all), seed=cfg["project"]["seed"])
    X = part[["x_m", "y_m"]].to_numpy("float64")
    for eps, (kind, v) in CONFIGS:
        ms = min_samples_from_frac(len(X), v) if kind == "frac" else v
        r = {"n": len(X), "eps_m": eps, "min_samples": ms, "ms_rule": f"{v * 100:g}%" if kind == "frac" else f"spec {v}"}
        pairs = estimate_full_neighbors(X, eps)
        r["est_pairs"] = pairs; r["est_gb"] = pairs * 8 / 1e9
        (lw, dbw), r["weighted_sec"], _, r["weighted_mem_mb"] = measure(lambda: fit_weighted(X, eps, ms))
        ram = system_ram_ratio()
        if r["est_gb"] > MAX_FULL_GB or ram > cfg["evaluation"]["ram_stop_ratio"]:
            r["full_status"] = f"bo qua (uoc tinh {r['est_gb']:.1f} GB, RAM he thong {ram:.0%})"
        else:
            lf, r["full_sec"], _, r["full_mem_mb"] = measure(lambda: DBSCAN(eps=eps, min_samples=ms).fit_predict(X))
            r.update(full_status="da chay", ari=adjusted_rand_score(lf, lw),
                     same_noise=bool(np.array_equal(lf == -1, lw == -1)),
                     n_clusters_full=len(set(lf) - {-1}), noise_full=float((lf == -1).mean()))
        r.update(n_clusters_weighted=len(set(lw) - {-1}), noise_weighted=float((lw == -1).mean()))
        rows.append(r)
        print(f"n={len(X):>6,} eps={eps:>4} ms={ms:>4}: {r['full_status']:<40s} "
              f"full {r.get('full_sec', float('nan')):6.2f}s / {r.get('full_mem_mb', float('nan')):7.1f} MB | "
              f"weighted {r['weighted_sec']:.3f}s / {r['weighted_mem_mb']:.1f} MB | ARI {r.get('ari', float('nan')):.4f}")
cmp42 = pd.DataFrame(rows).assign(time_slice="WEEKEND", sample_id=SAMPLE_ID)
p42 = resolve_path(cfg, "run_log").parent / f"dbscan_full_vs_weighted_{SAMPLE_ID}.csv"
cmp42.to_csv(p42, index=False)
print("Da luu:", p42.relative_to(cfg["_root"]))

In [ ]:
ran = cmp42[cmp42.full_status == "da chay"]
print("Tat ca lan chay full: ARI = 1 va tap nhieu trung khop:", bool((ran.ari == 1).all() and ran.same_noise.all()),
      f"({len(ran)}/{len(cmp42)} lan chay full)")
cmp42[["n", "eps_m", "min_samples", "ms_rule", "est_gb", "full_status", "full_sec", "full_mem_mb",
       "weighted_sec", "weighted_mem_mb", "ari", "n_clusters_full", "n_clusters_weighted", "noise_full", "noise_weighted"]].round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for eps, (kind, v) in CONFIGS:
    d = cmp42[cmp42.eps_m == eps]
    lab = f"eps {eps}, {d.ms_rule.iloc[0]}"
    l, = axes[0].plot(d.n, d.full_sec, "o-", label=f"full - {lab}")
    axes[0].plot(d.n, d.weighted_sec, "s--", color=l.get_color(), label=f"weighted - {lab}")
    axes[1].plot(d.n, d.full_mem_mb, "o-", color=l.get_color(), label=f"full - {lab}")
    axes[1].plot(d.n, d.est_gb * 1000, ":", color=l.get_color(), alpha=.6, label=f"uoc tinh chi so - {lab}")
axes[0].set_yscale("log"); axes[0].set_ylabel("Thoi gian fit (s, thang log)")
axes[1].set_ylabel("RAM them khi fit (MB)")
for ax in axes:
    ax.set_xlabel("So chuyen (WEEKEND)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
fig.suptitle(f"DBSCAN: chay tren tung chuyen (full) vs toa do co trong so (weighted) - {SAMPLE_ID}")
fig.tight_layout()
fig.savefig(resolve_path(cfg, "figures_dir") / f"dbscan_full_vs_weighted_{SAMPLE_ID}.png", dpi=150)

## 4.3 Lưới eps × min_samples

- eps lấy từ config (chốt sau 4.1): 500, 750, 1000, 1250, 1500, 2000, 2500, 3000 m.
- `min_samples`: 5 và 20 (spec), cộng 0,1 / 0,3 / 0,5 / 1% số chuyến của lát cắt.
- Chạy theo cách `weighted` (đã chứng minh trùng với `full` ở 4.2). Chỉ số tính trên **từng chuyến** bằng `evaluate`, giống K-Means.
- Thêm `silhouette_unique` (tính trên các toạ độ duy nhất không phải nhiễu) và `noise_coords` (số toạ độ bị gán nhiễu).

In [ ]:
from src.models.dbscan import scan_grid

def ms_rules(n_trips):
    return ([(f"spec {m}", m) for m in DB["min_samples"]] +
            [(f"{f * 100:g}%", min_samples_from_frac(n_trips, f)) for f in DB["min_samples_frac"]])

RERUN_GRID = False   # True: chay lai ca luoi (~11 phut, chu yeu do tinh Silhouette); False: doc CSV neu da co
tp = resolve_path(cfg, "run_log").parent / f"dbscan_tuning_{SAMPLE_ID}.csv"
if tp.exists() and not RERUN_GRID:
    tuning = pd.read_csv(tp)
    print("Doc ket qua luoi da luu:", tp.relative_to(cfg["_root"]), f"({len(tuning)} dong). Dat RERUN_GRID = True de chay lai.")
else:
    grids = []
    for s in SLICES:
        X = df.loc[df.time_slice == s, ["x_m", "y_m"]].to_numpy("float64")
        g = scan_grid(X, DB["eps_m"], ms_rules(len(X)), seed=cfg["project"]["seed"],
                      silhouette_max_sample=cfg["evaluation"]["silhouette_max_sample"])
        grids.append(g.assign(time_slice=s, sample_id=SAMPLE_ID, mode="weighted"))
        print(f"{s:11s} {len(g)} cau hinh, fit tong {g.fit_sec.sum():.2f}s, RAM them toi da {g.mem_delta_mb.max():.1f} MB")
    tuning = pd.concat(grids, ignore_index=True)
    tuning.to_csv(tp, index=False)
    print("Da luu:", tp.relative_to(cfg["_root"]), f"({len(tuning)} dong)")

### Heatmap theo lát cắt

Hàng là quy tắc `min_samples`, cột là eps (m). Ô trống nghĩa là không tính được chỉ số (ít hơn 2 cụm).

In [ ]:
RULE_ORDER = [r for r, _ in ms_rules(1000)]
panels = [("n_clusters", "So cum", "{:.0f}", "viridis"), ("noise_ratio", "Noise Ratio (%)", "{:.0f}", "Greys"),
          ("silhouette", "Silhouette (tung chuyen)", "{:.2f}", "RdYlGn"), ("largest_share", "Ty trong cum lon nhat (%)", "{:.0f}", "Oranges")]
fig, axes = plt.subplots(len(SLICES), len(panels), figsize=(4.6 * len(panels), 3.4 * len(SLICES)))
for i, s in enumerate(SLICES):
    t = tuning[tuning.time_slice == s]
    for j, (col, title, fmt, cmap) in enumerate(panels):
        mat = t.pivot(index="ms_rule", columns="eps_m", values=col).reindex(RULE_ORDER).astype(float)
        if col in ("noise_ratio", "largest_share"):
            mat = mat * 100
        ax = axes[i, j]
        im = ax.imshow(mat.values, cmap=cmap, aspect="auto")
        for (a, b), v in np.ndenumerate(mat.values):
            if np.isfinite(v):
                r, g_, b_, _ = im.cmap(im.norm(v))          # chu trang tren nen toi de doc duoc
                ax.text(b, a, fmt.format(v), ha="center", va="center", fontsize=7,
                        color="white" if 0.299 * r + 0.587 * g_ + 0.114 * b_ < 0.5 else "black")
        ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels([f"{e:.0f}" for e in mat.columns], fontsize=7, rotation=45)
        ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels(mat.index, fontsize=7)
        ax.set_title(f"{s} - {title}", fontsize=9)
        if i == len(SLICES) - 1: ax.set_xlabel("eps (m)")
fig.suptitle(f"DBSCAN luoi eps x min_samples - {SAMPLE_ID} (weighted)")
fig.tight_layout()
fig.savefig(resolve_path(cfg, "figures_dir") / f"dbscan_tuning_{SAMPLE_ID}.png", dpi=130)

### Lọc sơ bộ các cấu hình dùng được

Chỉ để **thu hẹp** trước khi chọn ở 4.4, không tự động chọn. Ngưỡng mô tả:
- 3–15 cụm (đủ để có hotspot, không vụn);
- Noise Ratio 2–25% (có lọc nhiễu nhưng không bỏ quá nhiều chuyến);
- cụm lớn nhất ≤ 80% (không gộp cả thành phố thành một cụm).

In [ ]:
ok = tuning[tuning.n_clusters.between(3, 15) & tuning.noise_ratio.between(0.02, 0.25) & (tuning.largest_share <= 0.8)]
print("So cau hinh qua loc theo lat cat:", ok.groupby("time_slice").size().to_dict())
print("Quy tac min_samples co cau hinh qua loc:", ok.groupby("ms_rule").size().to_dict())
cols = ["time_slice", "eps_m", "ms_rule", "min_samples", "n_clusters", "noise_ratio", "noise_coords",
        "largest_share", "silhouette", "silhouette_unique", "davies_bouldin"]
ok.sort_values(["time_slice", "silhouette"], ascending=[True, False])[cols].round(3)

## 4.4 Chọn cấu hình: so sánh các ứng viên

Bốn ứng viên lấy từ vùng hợp lý ở 4.3. Mỗi ứng viên dùng **cùng một quy tắc `min_samples` và cùng eps cho cả 3 lát cắt**, để so sánh được giữa các lát cắt (giống K = 5 của K-Means).

- A: 1%, eps 1500 m
- B: 1%, eps 2000 m
- C: 0,5%, eps 1000 m
- D: 0,5%, eps 1250 m

In [ ]:
CANDS = {"A": ("1%", 1500.0), "B": ("1%", 2000.0), "C": ("0.5%", 1000.0), "D": ("0.5%", 1250.0)}
rows = []
for name, (rule, eps) in CANDS.items():
    t = tuning[(tuning.ms_rule == rule) & (tuning.eps_m == eps)].set_index("time_slice").reindex(SLICES)
    for s in SLICES:
        r = t.loc[s]
        rows.append({"cand": name, "rule": rule, "eps_m": eps, "time_slice": s, "min_samples": int(r.min_samples),
                     "n_clusters": int(r.n_clusters), "noise_pct": r.noise_ratio * 100, "noise_coords": int(r.noise_coords),
                     "largest_pct": r.largest_share * 100, "silhouette": r.silhouette,
                     "silhouette_unique": r.silhouette_unique, "davies_bouldin": r.davies_bouldin})
cand_tab = pd.DataFrame(rows)
print("Trung binh 3 lat cat:")
print(cand_tab.groupby("cand")[["n_clusters", "noise_pct", "largest_pct", "silhouette", "silhouette_unique", "davies_bouldin"]]
      .mean().round(3).to_string())
cand_tab.round(3)

### Bản đồ tĩnh các ứng viên

Mỗi chấm là một toạ độ, kích thước theo số chuyến, màu là cụm (đánh số theo kích thước). **Dấu x xám là nhiễu.**

In [ ]:
from src.visualization.maps import PALETTE
from src.evaluation.hotspots import relabel_by_size
from src.models.dbscan import fit_weighted as db_fit_weighted
fig, axes = plt.subplots(len(CANDS), len(SLICES), figsize=(4.6 * len(SLICES), 4.4 * len(CANDS)))
for i, (name, (rule, eps)) in enumerate(CANDS.items()):
    for j, s in enumerate(SLICES):
        X = df.loc[df.time_slice == s, ["x_m", "y_m"]].to_numpy("float64")
        ms = int(cand_tab[(cand_tab.cand == name) & (cand_tab.time_slice == s)].min_samples.iloc[0])
        Xu, w, _ = aggregate_unique(X)
        _, db = db_fit_weighted(X, eps, ms)
        lu = relabel_by_size(db.labels_)
        ax = axes[i, j]
        noise = lu == -1
        ax.scatter(Xu[noise, 0] / 1000, Xu[noise, 1] / 1000, marker="x", s=12 + 300 * w[noise] / w.max(), c="#999999", linewidths=.8)
        ax.scatter(Xu[~noise, 0] / 1000, Xu[~noise, 1] / 1000, s=5 + 600 * w[~noise] / w.max(), c=[PALETTE[int(v) % len(PALETTE)] for v in lu[~noise]], alpha=.75, edgecolors="k", linewidths=.2)
        r = cand_tab[(cand_tab.cand == name) & (cand_tab.time_slice == s)].iloc[0]
        ax.set_title(f"{name}: {rule}, eps {eps:.0f} m - {s}" + chr(10) + f"{r.n_clusters} cum, nhieu {r.noise_pct:.1f}%, lon nhat {r.largest_pct:.0f}%", fontsize=9)
        ax.set_aspect("equal"); ax.grid(alpha=.3); ax.tick_params(labelsize=7)
fig.suptitle(f"DBSCAN - ung vien cau hinh - {SAMPLE_ID} (toa do x/y km, EPSG:26916)")
fig.tight_layout()
fig.savefig(resolve_path(cfg, "figures_dir") / f"dbscan_candidates_{SAMPLE_ID}.png", dpi=110)

## 4.4 Chạy chính thức cấu hình đã chốt

**Đã chốt trên `master_v1`** (Tân, 19/11/2026, mục 5.1; cập nhật từ B trên pilot):
- Cấu hình chính **A**: `min_samples` = 1% số chuyến, eps = 1500 m.
- Cấu hình phụ **C**: 0,5%, eps = 1000 m, dùng để minh hoạ độ nhạy của eps.

Hai cấu hình lấy từ `config: dbscan.final` và `dbscan.secondary`. Mỗi lát cắt × mỗi cấu hình chạy qua `run_and_log`: lưu nhãn, ghi một dòng vào `run_log.csv`, cột `note` ghi rõ chính hay phụ. Cụm được đánh số theo kích thước; nhiễu giữ nhãn -1.

**Mỗi lần chạy ô dưới sẽ ghi thêm 6 dòng vào run_log.csv** (chỉ thêm, có chủ đích).

In [ ]:
from src.evaluation import run_and_log, read_run_log
from src.evaluation.hotspots import relabel_by_size, cluster_table

ROLES = {"chinh (A)": DB["final"], "phu (C)": DB["secondary"]}
final_db = {}
for role, conf in ROLES.items():
    for s in SLICES:
        part = df[df.time_slice == s].reset_index(drop=True)
        X = part[["x_m", "y_m"]].to_numpy("float64")
        ms = min_samples_from_frac(len(X), conf["min_samples_frac"])
        eps = float(conf["eps_m"])
        params = {"eps_m": eps, "min_samples": ms, "min_samples_frac": conf["min_samples_frac"],
                  "mode": "weighted", "metric": DB["metric"], "role": role}
        row = run_and_log(cfg, "dbscan", lambda X: relabel_by_size(db_fit_weighted(X, eps, ms)[0]),
                          X, part.trip_id.to_numpy(), member="M2", time_slice=s, sample_id=SAMPLE_ID,
                          params=params, note=f"du lieu M1 {SAMPLE_ID}; cau hinh {role} chot o muc 5.1")
        lab = pd.read_parquet(resolve_path(cfg, "labels_dir") / f"{row['run_id']}.parquet")
        assert (lab.trip_id.values == part.trip_id.values).all()
        final_db[(role, s)] = (part, lab.label.to_numpy(), row)
        print(f"{role:9s} {row['run_id']}: {row['n_clusters']} cum, nhieu {row['noise_ratio']:.1%}, "
              f"Silhouette {row['silhouette']:.3f}, DBI {row['davies_bouldin']:.3f}, fit {row['runtime_fit_sec']:.3f}s")

### Bảng cụm

In [ ]:
tabs = []
for (role, s), (part, lab, row) in final_db.items():
    ct = cluster_table(part, lab, crs=cfg["project"]["crs"])
    tabs.append(ct.assign(role=role, time_slice=s, run_id=row["run_id"], sample_id=SAMPLE_ID,
                          noise_trips=int((lab == -1).sum()), noise_share=float((lab == -1).mean())))
db_clusters = pd.concat(tabs, ignore_index=True)
clp = resolve_path(cfg, "run_log").parent / f"dbscan_clusters_{SAMPLE_ID}.csv"
db_clusters.to_csv(clp, index=False)
print("Da luu:", clp.relative_to(cfg["_root"]))
show = db_clusters.assign(share_pct=(db_clusters.share * 100).round(1), radius_km=(db_clusters.radius_rms_m / 1000).round(2),
                          noise_pct=(db_clusters.noise_share * 100).round(1))
show[show.role == "chinh (A)"][["time_slice", "cluster", "n_trips", "share_pct", "n_coords", "radius_km",
                                "top_community_area", "noise_pct"]]

### Bản đồ tương tác

6 lớp: cấu hình chính A và phụ C cho 3 lát cắt. Điểm nhiễu màu xám. File HTML cần Internet để tải nền bản đồ.

In [ ]:
from src.visualization.maps import cluster_map
map_path = resolve_path(cfg, "maps_dir") / f"dbscan_map_{SAMPLE_ID}.html"
layers = {}
for (role, s), (part, lab, row) in final_db.items():
    p = row["params"] if isinstance(row["params"], dict) else json.loads(row["params"])
    layers[f"{role} {s} (eps {p['eps_m']:.0f} m, ms {p['min_samples']})"] = (
        part, lab, db_clusters[(db_clusters.role == role) & (db_clusters.time_slice == s)])
cluster_map(layers, map_path, title=f"DBSCAN - {SAMPLE_ID}",
            note=f"Chinh A: 1%, eps 1500 m; phu C: 0,5%, eps 1000 m. Du lieu chinh thuc M1 ({SAMPLE_ID}, 4 tuan 17/04-14/05/2023).")
print("Da luu:", map_path.relative_to(cfg["_root"]), f"({map_path.stat().st_size / 1024:.0f} KB)")

### Các dòng vừa ghi vào run_log.csv

In [ ]:
log = read_run_log(resolve_path(cfg, "run_log"))
ids = [row["run_id"] for _, _, row in final_db.values()]
log[log.run_id.isin(ids)][["run_id", "n_points", "n_clusters", "noise_ratio", "silhouette", "davies_bouldin",
                           "largest_share", "runtime_fit_sec", "mem_delta_mb", "note"]]